# Лабораторна робота 4. Криптосистема RSA

## Л4.1 Реалiзуйте тест на простоту Мiллера-Рабiна. Для кожного тестового числа $n$ бiнарної довжини 512, або проведiть 100 тестувань, в яких отримується вiдповiдь “можливо просте”, або отримайте вiдповiдь “складене”. Для кожного тестування виведiть вiдповiдне випадкове число $a$.

In [1]:
import random

def miller_rabin_test(n, k=100):
    if n <= 1: return "складене"
    if n <= 3: return "можливо просте"
    if n % 2 == 0: return "складене"
        
    # представлення n - 1 = 2^t * m
    m = n - 1
    t = 0
    
    while m % 2 == 0:
        # поки m парне, ділимо його на 2 і збільшуємо лічильник степеня t
        m //= 2
        t += 1
        
    for i in range(1, k + 1):
        a = random.randint(2, n - 2)
        print(f"Тестування {i}: a = {a}") 
        # u = a^d mod n
        u = pow(a, m, n)
        # якщо перша перевірка дала 1 або n-1, це "a" нам нічого не скаже, бо число "пройшло" цей раунд, і ми йдемо до наступного "a"
        if u == 1 or u == n - 1:
            continue            
        is_composite = True
        for _ in range(t - 1):
            # u = u^2 mod n
            # n-1 mod n == -1 mod n
            u = pow(u, 2, n)
            if u == n - 1:
                is_composite = False
                break
        if is_composite:
            return f"Результат: складене (на ітерації {i})"
    return f"Результат: можливо просте (після {k} тестувань)"
    
count_of_tests = 5 

for test_num in range(1, count_of_tests + 1):
    n_candidate = random.getrandbits(512) | 1
    
    print(f"\n{'='*30}")
    print(f"ТЕСТ №{test_num}")
    print(f"Перевіряємо n: {n_candidate}")
    print(f"{'='*30}")
    result = miller_rabin_test(n_candidate, 100)
    print(result)

# n_prime = 2**89 - 1 
# print(miller_rabin_test(n_prime, 100))


ТЕСТ №1
Перевіряємо n: 5604220855407954106997117524732661786337477156086936309009909327294567793486935284382960149782751716557563543039903052367299757885423875394506461821024387
Тестування 1: a = 1983067386240535230045462243267510189887019267969354093602763055720020290960361300840758109940437900884481214314707185198593532445557145297121784073346518
Результат: складене (на ітерації 1)

ТЕСТ №2
Перевіряємо n: 1430682075532602244177194726403923755060409974862011918873646601307141604567613447444757151375811934632868313452643488683930089251340001414987608204123067
Тестування 1: a = 1171765696878740873004923199863927831621324684514235687562608315156808031056985241785034625035287348718655920724194283237299417690229341494424578969787438
Результат: складене (на ітерації 1)

ТЕСТ №3
Перевіряємо n: 39057410412927691011394564654039408548397066273825415527692169409805020257914593779240071329729604023817088335850710481846968196185602232690798532269633
Тестування 1: a = 13924768098703899804373446864

## Л4.2 Реалiзуйте криптосистему RSA для секретних модулiв бiнарної довжини 512. Для дешифрування використайте китайську теорему про остачi.

In [2]:
def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

def egcd(a, b):
    if a == 0:
        return (b, 0, 1)
    else:
        gcd, x1, y1 = egcd(b % a, a)
        x = y1 - (b // a) * x1
        y = x1
    return gcd, x, y

def modinv(a, m):
    g, x, y = egcd(a, m)
    if g != 1:
        return None
    else:
        return x % m

def miller_rabin_test(n, k=100):
    if n <= 1: return False
    if n <= 3: return True
    if n % 2 == 0: return False
        
    m = n - 1
    t = 0
    while m % 2 == 0:
        m //= 2
        t += 1
        
    for i in range(1, k + 1):
        a = random.randint(2, n - 2)
        u = pow(a, m, n)
        if u == 1 or u == n - 1:
            continue             
        is_composite = True
        for _ in range(t - 1):
            u = pow(u, 2, n)
            if u == n - 1:
                is_composite = False
                break
        if is_composite:
            return False
    return True


def chinese_decr(c, p, q, dp, dq, qinv):
    """
    c: шифротекст
    p, q: секретні прості модулі
    dp, dq: секретні експоненти для кожного модуля (d mod p-1 та d mod q-1)
    qinv: коефіцієнт CRT
    """
    m1 = pow(c, dp, p)  # m1 = c^dp mod p
    m2 = pow(c, dq, q)  # m2 = c^dq mod q
    # Формула Гарнера для CRT
    h = (qinv * (m1 - m2 + p)) % p
    m = m2 + h * q
    return m

def get_prime_512():
    while True:
        p = random.getrandbits(512) | 1
        if miller_rabin_test(p, 100): 
            return p

def generate_keys():
    """Генерує пару ключів RSA та параметри для CRT."""
    p = get_prime_512()
    q = get_prime_512()
    
    while p == q:
        q = get_prime_512()

    if q > p:
        p, q = q, p

    n = p * q
    phi = (p - 1) * (q - 1)

    # найчатсіше RSA Public Key = 2^16 + 1 (число Ферма F4)
    e = 65537
    d = modinv(e, phi)

    # обчислюємо компоненти приватного ключа для CRT (RFC 8017)
    dP = d % (p - 1)
    dQ = d % (q - 1)
    qInv = modinv(q, p)

    return (e, n), (p, q, dP, dQ, qInv)

def encrypt(m, public_key):
    e, n = public_key
    return pow(m, e, n)

In [3]:
message = 1234567890123456789
print(f"Початкове повідомлення: {message}")

print("\nКрок 1. Генеруємо ключі (512-бітні p та q)")
pub, priv = generate_keys()
e, n = pub
p, q, dP, dQ, qInv = priv

print(f"Публічний ключ (e, n) згенеровано.")
print(f"n = p * q (секретні прості числа p та q) : {hex(n)}")

print("\nКрок 2. Шифруємо повідомлення")
c = encrypt(message, pub)
print(f"Шифротекст: {hex(c)}")

print("\nКрок 3. Дешифруємо через CRT")
decoded = chinese_decr(c, p, q, dP, dQ, qInv)

print(f"Дешифроване повідомлення: {decoded}")
print(f"\nРезультат правильний? {'ТАК' if message == decoded else 'НІ'}")

Початкове повідомлення: 1234567890123456789

Крок 1. Генеруємо ключі (512-бітні p та q)
Публічний ключ (e, n) згенеровано.
n = p * q (секретні прості числа p та q) : 0x7bfb31296ef2827b2393bd63de229dc76005db5a983797ad5af54bed7a049118feb7e448e649e087d0283daf501b769187b4a71723ba18660222408cfd89c7c376f8e20f1331595c1aae14c74fdac5008a55a50e4c62e525f04f62690d2125a0dfeae15513642352d03dd9acefddf05d77d5efd4973c479a48edfcd146a1987

Крок 2. Шифруємо повідомлення
Шифротекст: 0x6b3f739b6130f50cfda79f9f46c9ea91f5e6fc069a7c5aaf7711d0b041d5541526994043e378ac9bbffd6dbd8e26f65413b37b0f967e62898dfa6080f1c07a700671218807705e39f6e306ce07f94fe4666fd666993128ec3c7438ad08c59d3130a8b99845060a21e44cdd057bb8cd6324517f4b4e5fa34ba943649025efdae

Крок 3. Дешифруємо через CRT
Дешифроване повідомлення: 1234567890123456789

Результат правильний? ТАК


## Л4.3 Реалiзуйте RSA-OAEP.

Ключові компоненти

- MGF1 — функція генерації маски (розтягує хеш до потрібної довжини)
- Hash — хеш-функція (SHA-256)
- hLen — довжина виходу хешу в байтах
- k — довжина модуля RSA в байтах
- M — повідомлення, L — мітка (зазвичай порожня)

In [4]:
import hashlib
import os


def xor_bytes(b1, b2):
    # Функція для XOR байтів (беремо по одному байту і міняємо їх)
    # zip бере пари байтів: (перший з b1, перший з b2), (другий з b1, другий з b2)...
    # a ^ b — робимо XOR між цифрами
    # bytes(...) — збираємо результат назад у байтовий рядок
    return bytes(a ^ b for a, b in zip(b1, b2))


def mgf1(seed, length):
    """
    seed: маленьке випадкове число (зерно)
    length: якої довжини маску (шум) нам треба отримати
    """
    h_len = 32
    # порожній байтовий рядок
    mask = b""
    counter = 0
    
    # Ми додаємо до seed лічильник і хешуємо, поки не назбираємо потрібну довжину маски
    while len(mask) < length:
        # Конвертуємо число counter у 4 байти
        c = counter.to_bytes(4, byteorder='big')
        # Додаємо хеш до нашої маски, digest() завершує процес хешування і повертає результат у вигляді байтового рядка
        mask += hashlib.sha256(seed + c).digest()
        counter += 1
        
    # Відрізаємо зайве, якщо маска вийшла довшою
    return mask[:length]


def oaep_encode(message, k, label=b""):
    """
    message: повідомлення (байти)
    k: розмір RSA ключа в байтах 
    """
    h_len = 32
    
    # Створюємо хеш мітки (зазвичай порожньої)
    l_hash = hashlib.sha256(label).digest()
    
    # Рахуємо, скільки нулів треба додати для пакування
    ps_len = k - len(message) - 2 * h_len - 2
    # Число нуль, записане у форматі одного байта
    ps = b"\x00" * ps_len
    
    # Склеюємо: хеш + нулі + розділювач 0x01 + саме повідомлення
    db = l_hash + ps + b"\x01" + message
    
    # os.urandom — генерує випадкові байти з "шуму" процесора
    seed = os.urandom(h_len)
    
    # КРОК 1: Маскуємо DB
    # Створюємо шум такої ж довжини, як DB, використовуючи seed
    db_mask = mgf1(seed, len(db))
    # Накладаємо шум на дані
    masked_db = xor_bytes(db, db_mask)
    
    # КРОК 2: Маскуємо Seed
    # Створюємо шум для самого seed, використовуючи вже замасковані дані
    seed_mask = mgf1(masked_db, h_len)
    # Накладаємо шум на seed
    masked_seed = xor_bytes(seed, seed_mask)
    
    # Перший байт завжди 0x00 за стандартом
    return b"\x00" + masked_seed + masked_db


def oaep_decode(em, k, label=b""):
    h_len = 32
    
    # Розрізаємо отриманий блок на частини
    # em[0] — це перший нульовий байт
    masked_seed = em[1 : h_len + 1]
    masked_db = em[h_len + 1 :]
    
    # Генеруємо маску з masked_db
    seed_mask = mgf1(masked_db, h_len)
    seed = xor_bytes(masked_seed, seed_mask)
    
    # Тепер за допомогою seed знімаємо маску з даних
    db_mask = mgf1(seed, len(masked_db))
    db = xor_bytes(masked_db, db_mask)
    
    # Перевіряємо, чи все правильно
    l_hash = hashlib.sha256(label).digest()
    l_hash_prime = db[:h_len] # Хеш мітки, який ми знайшли всередині
    
    # Шукаємо, де закінчуються нулі і починається повідомлення
    # Розділювач 0x01 підкаже нам початок повідомлення
    remainder = db[h_len:]
    if b"\x01" not in remainder:
        raise ValueError("Помилка: Розділювач 0x01 не знайдено!")
        
    # Знаходимо індекс 0x01 і беремо все, що ПІСЛЯ нього
    m_start = remainder.find(b"\x01") + 1
    message = remainder[m_start:]
    
    # Перевірка: чи не підмінили нам дані?
    if l_hash != l_hash_prime:
        raise ValueError("Помилка: Хеші міток не збігаються!")
        
    return message

In [5]:
message = b"cryptography"

k = 128 

print(f"Оригінальне повідомлення: {message}")

try:
    encoded = oaep_encode(message, k)
    print(f"\nЗапаковане повідомлення: {encoded.hex()}")

    decoded = oaep_decode(encoded, k)
    print(f"\nРозпаковане повідомлення: {decoded}")
    
    if message == decoded:
        print("\nПеревірка успішна! OAEP працює правильно.")
    else:
        print("\nПомилка: дані пошкоджено.")

except ValueError as e:
    print(f"\nСталася помилка: {e}")

Оригінальне повідомлення: b'cryptography'

Запаковане повідомлення: 00afb0ba5b5d4f13a22c97d9ee1d1786119758cb301574a5522a124a2b864a04e2a1d58adf1f72db8e0ad2af74b68a72997f92e913a682ba57a5a830f2252746b702d1097a38dc77850a427a14026cc164a7f2e93161aef6f86ed4d3d8c92249b2500595f94653e74e155932e88f719050af3736423ec11c4990336b35aab38e

Розпаковане повідомлення: b'cryptography'

Перевірка успішна! OAEP працює правильно.
